# Phase 3: 3-Class Clone Separation Lab (`glm-5.3`, `glm-5.3-flash`, `claude-fable-5`)

**目的**:
表層 N-gram では分離できないクローン文体 3 クラスに対し、外部拡張データ（`prvsiyan/prvsiyans-extended-name-that-llm-dataset`）を用いたデータ増強と深層表現（Qwen3 中間層プロービング + DeBERTa-v3-large 専用ヘッド学習）を行い、**2-Passage Sparse Sampling OOF 上で統計的有意差（95% 信頼区間 > 0）** が出るか検証します。

**厳格なリーク防止ルール**:
1. 検証（Validation）は**必ず公式 `train.csv` のみ**（5-Fold GroupKFold by `prompt_id`）で行う。
2. 拡張データは公式 `train.csv` とテキスト重複しない行のみに絞り込み、各 Fold の **学習（Train）側のみ** に追加する。


In [ ]:
# [Cell 1] Environment & Kaggle API Setup
!pip install -q transformers accelerate datasets scikit-learn scipy kaggle bitsandbytes

import os, sys, json, re, gc, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy.special import softmax
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print(f"Kaggle Authenticated via Colab Secrets: {os.environ['KAGGLE_USERNAME']}")
except Exception:
    print("Using existing ~/.kaggle/kaggle.json or environment credentials.")

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# [Cell 2] Data Acquisition & Leak-Free 3-Class Construction
DATA_DIR = Path('/content/ntl_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / 'train.csv').exists():
    !kaggle competitions download -c name-that-llm -f train.csv -p /content/ntl_data
    !unzip -o -q /content/ntl_data/train.csv.zip -d /content/ntl_data

if not (DATA_DIR / 'ext').exists():
    (DATA_DIR / 'ext').mkdir(exist_ok=True)
    !kaggle datasets download -d prvsiyan/prvsiyans-extended-name-that-llm-dataset -p /content/ntl_data/ext --unzip

# Optional Open Pool Scaling (sadamtorres scaling law: up to 6,000-12,000 rows per class)
USE_OPEN_POOL = True
OPEN_POOL_PER_CLASS = 4000
if USE_OPEN_POOL and not (DATA_DIR / 'open_pool').exists():
    (DATA_DIR / 'open_pool').mkdir(exist_ok=True)
    !kaggle datasets download -d crtproduction/name-that-llm-open-pool -p /content/ntl_data/open_pool --unzip || true

# Download verified True OOF assets for final 2-Passage Cascade Gate
OOF_DIR = Path('/content/oof_assets')
OOF_DIR.mkdir(parents=True, exist_ok=True)
if not (OOF_DIR / 'q8_oof.npz').exists():
    !kaggle kernels output concyclics/ntl-native-q8-multidepth-fivefold-0679 -p /content/oof_assets/q8
    !kaggle datasets download -d dhruvtibarewal/nlm-matched-clean -p /content/oof_assets/misc --unzip || true

df_train = pd.read_csv(DATA_DIR / 'train.csv')
print(f"Official train.csv: {df_train.shape}")

# Load extended dataset (prefer passages_clean.csv if present)
ext_files = list((DATA_DIR / 'ext').glob('*.csv'))
print("Extended CSV files:", [f.name for f in ext_files])
ext_path = DATA_DIR / 'ext' / 'passages_clean.csv'
if not ext_path.exists():
    ext_path = ext_files[0]
df_ext = pd.read_csv(ext_path)
print(f"Extended dataset ({ext_path.name}): {df_ext.shape}")

TARGET_NAMES = ['glm-5.3', 'glm-5.3-flash', 'claude-fable-5']
TARGET_MAP = {name: i for i, name in enumerate(TARGET_NAMES)}

# Normalize text helper for strict deduplication
def norm_txt(s):
    return re.sub(r'\s+', ' ', str(s).strip().lower())

official_norms = set(df_train['text'].map(norm_txt))
df_ext_3 = df_ext[df_ext['model'].isin(TARGET_NAMES)].copy()

if USE_OPEN_POOL and (DATA_DIR / 'open_pool').exists():
    op_files = list((DATA_DIR / 'open_pool').glob('*.parquet')) + list((DATA_DIR / 'open_pool').glob('*.csv'))
    if op_files:
        op_df = pd.read_parquet(op_files[0]) if op_files[0].suffix == '.parquet' else pd.read_csv(op_files[0])
        if 'model' in op_df.columns and 'text' in op_df.columns:
            op_3 = op_df[op_df['model'].isin(TARGET_NAMES)].copy()
            op_3 = op_3.groupby('model', group_keys=False).apply(lambda g: g.sample(min(len(g), OPEN_POOL_PER_CLASS), random_state=42))
            df_ext_3 = pd.concat([df_ext_3[['model', 'text']], op_3[['model', 'text']]], ignore_index=True)
            print(f"Merged Open Pool 3-class samples (+{len(op_3)} rows)")

df_ext_3['norm'] = df_ext_3['text'].map(norm_txt)
before_dedup = len(df_ext_3)
df_ext_3 = df_ext_3[~df_ext_3['norm'].isin(official_norms)].drop_duplicates('norm').reset_index(drop=True)
print(f"Extended 3-class rows: {before_dedup} -> {len(df_ext_3)} after strict dedup vs train.csv")
print(df_ext_3['model'].value_counts())

df_off_3 = df_train[df_train['model'].isin(TARGET_NAMES)].copy().reset_index(drop=True)
print(f"Official 3-class rows: {len(df_off_3)}")
print(df_off_3['model'].value_counts())


In [ ]:
# [Cell 3] Experiment 1: Qwen3-Embedding-8B Multi-Layer Probing (Layers 8, 12, 16, 20, 24, Last)
# Note: Extracts hidden states without fine-tuning to check which layer separates the 3 clones best.
from transformers import AutoTokenizer, AutoModel

MODEL_ID = "Qwen/Qwen3-Embedding-8B"
PROBE_LAYERS = [8, 12, 16, 20, 24, -1]
BATCH_SIZE = 16
MAX_LEN = 384

print(f"Loading {MODEL_ID} in float16 (or 4-bit if VRAM < 20GB)...")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
load_kwargs = {"torch_dtype": torch.float16, "device_map": "auto"}
if vram_gb < 20:
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
model = AutoModel.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()

def extract_layer_embeddings(texts, layers=PROBE_LAYERS, batch_size=BATCH_SIZE):
    all_vecs = {l: [] for l in layers}
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_txt = list(texts[i:i+batch_size])
            enc = tokenizer(batch_txt, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(model.device)
            out = model(**enc, output_hidden_states=True, use_cache=False)
            hs = out.hidden_states
            mask = enc['attention_mask'].unsqueeze(-1).float()
            for l in layers:
                h = hs[l].float()
                # Mean pooling over non-pad tokens
                pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
                # L2 normalize
                pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
                all_vecs[l].append(pooled.cpu().numpy().astype(np.float32))
    return {l: np.vstack(v) for l, v in all_vecs.items()}

print("Extracting embeddings for ALL official train.csv (2,699 rows)...")
t0 = time.time()
vecs_off_all = extract_layer_embeddings(df_train['text'].values)
print(f"Official extracted in {time.time()-t0:.1f}s")

print("Extracting embeddings for Extended 3-class rows...")
t0 = time.time()
vecs_ext_3 = extract_layer_embeddings(df_ext_3['text'].values)
print(f"Extended extracted in {time.time()-t0:.1f}s")

# Free VRAM
del model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# [Cell 4] Evaluate Layer-by-Layer 5-Fold GroupKFold Accuracy (Official vs Official+Extended)
off_3_mask = df_train['model'].isin(TARGET_NAMES).values
y_off_3 = df_train.loc[off_3_mask, 'model'].map(TARGET_MAP).values
pids_off_3 = df_train.loc[off_3_mask, 'prompt_id'].values
y_ext_3 = df_ext_3['model'].map(TARGET_MAP).values

gkf = GroupKFold(n_splits=5)

print("=== Layer Probing Results on Official 449 3-Class Samples ===")
print(f"{'Layer':<8} | {'Off-Only Acc':<14} | {'Off+Ext Acc':<14} | {'Best C'}")
print("-" * 55)

best_layer = None
best_acc = -1.0
best_oof_probs_all = None

for l in PROBE_LAYERS:
    X_off_3 = vecs_off_all[l][off_3_mask]
    X_ext_3 = vecs_ext_3[l]
    X_all_2699 = vecs_off_all[l]
    
    # 1. Official Only
    oof_only = np.zeros((len(y_off_3), 3))
    for tr_i, val_i in gkf.split(X_off_3, y_off_3, groups=pids_off_3):
        clf = LogisticRegression(C=2.0, max_iter=1000)
        clf.fit(X_off_3[tr_i], y_off_3[tr_i])
        oof_only[val_i] = clf.predict_proba(X_off_3[val_i])
    acc_only = accuracy_score(y_off_3, oof_only.argmax(1))
    
    # 2. Official + Extended (Extended added strictly to Train split)
    best_c_acc = -1.0
    best_c = 1.0
    best_probs_2699 = None
    for C in [0.5, 1.0, 2.0, 5.0]:
        oof_ext = np.zeros((len(y_off_3), 3))
        probs_2699 = np.zeros((len(df_train), 3))
        # Map val_i back to full 2699 indices using prompt_id
        for tr_i, val_i in gkf.split(X_off_3, y_off_3, groups=pids_off_3):
            X_tr = np.vstack([X_off_3[tr_i], X_ext_3])
            y_tr = np.concatenate([y_off_3[tr_i], y_ext_3])
            clf = LogisticRegression(C=C, max_iter=1000)
            clf.fit(X_tr, y_tr)
            oof_ext[val_i] = clf.predict_proba(X_off_3[val_i])
            
            # Predict on all 2,699 rows belonging to val prompts (strictly out-of-fold by prompt_id)
            val_pids = set(pids_off_3[val_i])
            full_val_mask = df_train['prompt_id'].isin(val_pids).values
            probs_2699[full_val_mask] = clf.predict_proba(X_all_2699[full_val_mask])
            
        acc_ext = accuracy_score(y_off_3, oof_ext.argmax(1))
        if acc_ext > best_c_acc:
            best_c_acc = acc_ext
            best_c = C
            best_probs_2699 = probs_2699
            
    print(f"{str(l):<8} | {acc_only:.4f}         | {best_c_acc:.4f}         | C={best_c}")
    if best_c_acc > best_acc:
        best_acc = best_c_acc
        best_layer = l
        best_oof_probs_all = best_probs_2699

print(f"\nBest Probing Layer: {best_layer} (3-Class Acc = {best_acc:.4f})")
np.save('/content/q8_probe_3class_oof_2699.npy', best_oof_probs_all)
print("Saved /content/q8_probe_3class_oof_2699.npy (download or evaluate in Cell 5).")


In [ ]:
# [Cell 5] Experiment 2: DeBERTa-v3-large 3-Class Fine-Tuning (5-Fold GroupKFold)
# Optional heavy run: Fine-tunes DeBERTa-v3-large specifically on the 3 clone classes.
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup

RUN_DEBERTA_FT = True
DEB_MODEL = "microsoft/deberta-v3-large"
EPOCHS = 3
LR = 1.5e-5
BS = 8

if RUN_DEBERTA_FT:
    deb_tok = AutoTokenizer.from_pretrained(DEB_MODEL)
    
    class TxtDS(Dataset):
        def __init__(self, texts, labels=None, max_len=384):
            self.texts = list(texts)
            self.labels = labels
            self.max_len = max_len
        def __len__(self): return len(self.texts)
        def __getitem__(self, idx):
            enc = deb_tok(self.texts[idx], truncation=True, max_length=self.max_len, padding='max_length', return_tensors='pt')
            item = {k: v.squeeze(0) for k, v in enc.items()}
            if self.labels is not None:
                item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
            return item

    oof_deb_3 = np.zeros((len(y_off_3), 3))
    probs_deb_2699 = np.zeros((len(df_train), 3))
    
    off_3_texts = df_train.loc[off_3_mask, 'text'].values
    ext_3_texts = df_ext_3['text'].values
    
    for fold, (tr_i, val_i) in enumerate(gkf.split(off_3_texts, y_off_3, groups=pids_off_3)):
        print(f"--- DeBERTa-v3-large Fold {fold+1}/5 ---")
        tr_texts = np.concatenate([off_3_texts[tr_i], ext_3_texts])
        tr_labels = np.concatenate([y_off_3[tr_i], y_ext_3])
        
        tr_dl = DataLoader(TxtDS(tr_texts, tr_labels), batch_size=BS, shuffle=True)
        val_dl = DataLoader(TxtDS(off_3_texts[val_i]), batch_size=BS*2, shuffle=False)
        
        val_pids = set(pids_off_3[val_i])
        full_val_mask = df_train['prompt_id'].isin(val_pids).values
        full_dl = DataLoader(TxtDS(df_train.loc[full_val_mask, 'text'].values), batch_size=BS*2, shuffle=False)
        
        clf_m = AutoModelForSequenceClassification.from_pretrained(DEB_MODEL, num_labels=3).cuda()
        opt = torch.optim.AdamW(clf_m.parameters(), lr=LR, weight_decay=0.01)
        sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=int(0.1*len(tr_dl)*EPOCHS), num_training_steps=len(tr_dl)*EPOCHS)
        
        for ep in range(EPOCHS):
            clf_m.train()
            for batch in tr_dl:
                opt.zero_grad()
                batch = {k: v.cuda() for k, v in batch.items()}
                out = clf_m(**batch)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(clf_m.parameters(), 1.0)
                opt.step()
                sched.step()
                
        clf_m.eval()
        val_preds = []
        with torch.no_grad():
            for batch in val_dl:
                batch = {k: v.cuda() for k, v in batch.items()}
                val_preds.append(torch.softmax(clf_m(**batch).logits, dim=-1).cpu().numpy())
        oof_deb_3[val_i] = np.vstack(val_preds)
        
        full_preds = []
        with torch.no_grad():
            for batch in full_dl:
                batch = {k: v.cuda() for k, v in batch.items()}
                full_preds.append(torch.softmax(clf_m(**batch).logits, dim=-1).cpu().numpy())
        probs_deb_2699[full_val_mask] = np.vstack(full_preds)
        
        fold_acc = accuracy_score(y_off_3[val_i], oof_deb_3[val_i].argmax(1))
        print(f"Fold {fold+1} 3-Class Acc: {fold_acc:.4f}")
        del clf_m, opt, sched
        gc.collect()
        torch.cuda.empty_cache()
        
    deb_acc = accuracy_score(y_off_3, oof_deb_3.argmax(1))
    print(f"\nOverall DeBERTa-v3-large 3-Class OOF Acc: {deb_acc:.4f}")
    np.save('/content/deberta_3class_oof_2699.npy', probs_deb_2699)
